Merge...

## 1. Carga de librerias

In [1]:
import pandas as pd
import numpy as np

## 2. Carga de los datsets limpios

In [2]:
padron = pd.read_csv("../data_processed/padron_limpio.csv")
edad = pd.read_csv("../data_processed/edad_limpio.csv")
nacionalidad = pd.read_csv("../data_processed/nacionalidad_limpio.csv")
superficie = pd.read_csv("../data_processed/superficie_limpio.csv")

## 3. Comprobación de estructura de los datasets

In [3]:
padron.shape, padron.columns

((235376, 4),
 Index(['codigo_municipio', 'municipio', 'periodo', 'poblacion'], dtype='object'))

In [4]:
edad.shape, edad.columns

((3409707, 6),
 Index(['codigo_municipio', 'municipio', 'periodo', 'edad_grupo', 'edad_inicio',
        'poblacion'],
       dtype='object'))

In [5]:
nacionalidad.shape, nacionalidad.columns

((324734, 5),
 Index(['codigo_municipio', 'municipio', 'periodo', 'nacionalidad',
        'poblacion'],
       dtype='object'))

In [6]:
superficie.shape, superficie.columns

((8132, 3),
 Index(['codigo_municipio', 'municipio', 'superficie_km2'], dtype='object'))

## 4. Crear variable de poblacion mayor de 65 años

In [7]:
edad_65 = edad[edad['edad_inicio'] >= 65].copy()

In [8]:
edad_65.head(8), edad_65.shape

(    codigo_municipio         municipio  periodo       edad_grupo  edad_inicio  \
 13              1001  Alegría-Dulantzi     2003  De 65 a 69 años           65   
 14              1001  Alegría-Dulantzi     2003  De 70 a 74 años           70   
 15              1001  Alegría-Dulantzi     2003  De 75 a 79 años           75   
 16              1001  Alegría-Dulantzi     2003  De 80 a 84 años           80   
 17              1001  Alegría-Dulantzi     2003  De 85 a 89 años           85   
 18              1001  Alegría-Dulantzi     2003  De 90 a 94 años           90   
 19              1001  Alegría-Dulantzi     2003  De 95 a 99 años           95   
 20              1001  Alegría-Dulantzi     2003   100 y más años          100   
 
     poblacion  
 13       54.0  
 14       39.0  
 15       30.0  
 16       31.0  
 17       11.0  
 18       11.0  
 19        5.0  
 20        0.0  ,
 (1298936, 6))

## 5. Calcular población total ≥65

In [9]:
# Tenemos varias filas por municipio-año, una por cada grupo de edad, vamos a sumarlas para obtener la pablación mayor de 65
mayores_65 = (
    edad_65
    .groupby(['codigo_municipio', 'municipio', 'periodo'], as_index=False)['poblacion']
    .sum()
)

In [10]:
# Renombramos la columna para que sea clara
mayores_65 = mayores_65.rename(columns={'poblacion': 'poblacion_mayor_65'})

In [11]:
mayores_65.head()

,codigo_municipio,municipio,periodo,poblacion_mayor_65
0,1001,Alegría-Dulantzi,2003,181.0
1,1001,Alegría-Dulantzi,2004,174.0
2,1001,Alegría-Dulantzi,2005,176.0
3,1001,Alegría-Dulantzi,2006,190.0
4,1001,Alegría-Dulantzi,2007,203.0


## 6. Crear dataset de población extranjera

In [12]:
# Primero filtramos solo la población extranjera
extranjeros = nacionalidad[nacionalidad['nacionalidad'] == "Extranjera"].copy()

In [13]:
extranjeros.head(), extranjeros.shape

(   codigo_municipio         municipio  periodo nacionalidad  poblacion
 1              1001  Alegría-Dulantzi     2003   Extranjera       30.0
 3              1001  Alegría-Dulantzi     2004   Extranjera       56.0
 5              1001  Alegría-Dulantzi     2005   Extranjera       79.0
 7              1001  Alegría-Dulantzi     2006   Extranjera       92.0
 9              1001  Alegría-Dulantzi     2007   Extranjera      114.0,
 (162367, 5))

## 7. Calcular población extranjera

In [14]:
# Calculamos población extranjera por municipio y año
extranjeros_mun = (
    extranjeros
    .groupby(['codigo_municipio', 'municipio', 'periodo'], as_index=False)['poblacion']
    .sum()
)

In [15]:
# Renombramos la columna
extranjeros_mun = extranjeros_mun.rename(columns={'poblacion': 'poblacion_extranjera'})

In [16]:
extranjeros_mun.head()

,codigo_municipio,municipio,periodo,poblacion_extranjera
0,1001,Alegría-Dulantzi,2003,30.0
1,1001,Alegría-Dulantzi,2004,56.0
2,1001,Alegría-Dulantzi,2005,79.0
3,1001,Alegría-Dulantzi,2006,92.0
4,1001,Alegría-Dulantzi,2007,114.0


## 8. Construcción dataset maestro

In [18]:
# Reiniciamos el dataframe meatro desde padrón
df = padron.copy()

In [19]:
# Añadimos población mayor de 65
df = df.merge(
    mayores_65[['codigo_municipio', 'periodo', 'poblacion_mayor_65']],
    on=['codigo_municipio', 'periodo'],
    how="left"
)

In [20]:
# Añadimos población extranjera
df = df.merge(
    extranjeros_mun[['codigo_municipio', 'periodo', 'poblacion_extranjera']],
    on=['codigo_municipio', 'periodo'],
    how="left"
)

In [21]:
# Añadimos superficie
df = df.merge(
    superficie[['codigo_municipio', 'superficie_km2']],
    on='codigo_municipio',
    how="left"
)

In [22]:
df.head(10), df.columns, df.shape

(   codigo_municipio         municipio  periodo  poblacion  poblacion_mayor_65  \
 0              1001  Alegría-Dulantzi     1996     1234.0                 NaN   
 1              1001  Alegría-Dulantzi     1998     1259.0                 NaN   
 2              1001  Alegría-Dulantzi     1999     1329.0                 NaN   
 3              1001  Alegría-Dulantzi     2000     1401.0                 NaN   
 4              1001  Alegría-Dulantzi     2001     1486.0                 NaN   
 5              1001  Alegría-Dulantzi     2002     1598.0                 NaN   
 6              1001  Alegría-Dulantzi     2003     1707.0               181.0   
 7              1001  Alegría-Dulantzi     2004     1919.0               174.0   
 8              1001  Alegría-Dulantzi     2005     2048.0               176.0   
 9              1001  Alegría-Dulantzi     2006     2189.0               190.0   
 
    poblacion_extranjera  superficie_km2  
 0                   NaN           19.95  
 1        

## 9. Filtrar por periodo común (2003-2022)

In [23]:
df = df[(df['periodo'] >= 2003) & (df['periodo'] <= 2022)]

In [24]:
df['periodo'].min(), df['periodo'].max()

(2003, 2022)

## 12. Calcular densidad de población

In [25]:
df['densidad_poblacion'] = df['poblacion'] / df['superficie_km2']

In [26]:
# Comprobamos 
df[['municipio', 'periodo', 'poblacion', 'superficie_km2', 'densidad_poblacion']].head()

,municipio,periodo,poblacion,superficie_km2,densidad_poblacion
6,Alegría-Dulantzi,2003,1707.0,19.95,85.563910
7,Alegría-Dulantzi,2004,1919.0,19.95,96.190476
8,Alegría-Dulantzi,2005,2048.0,19.95,102.656642
9,Alegría-Dulantzi,2006,2189.0,19.95,109.724311
10,Alegría-Dulantzi,2007,2305.0,19.95,115.538847


## 13. Porcentaje de población mayor de 65

In [27]:
df['porcentaje_mayor_65'] = df['poblacion_mayor_65'] / df['poblacion'] * 100

## 14. porcentaje de población extranjera

In [29]:
df['porcentaje_extranjeros'] = df['poblacion_extranjera'] / df['poblacion'] * 100

In [30]:
# Comprobamos
df[[
    'municipio',
    'periodo',
    'poblacion',
    'poblacion_mayor_65',
    'poblacion_extranjera',
    'porcentaje_mayor_65',
    'porcentaje_extranjeros'
]].head()

,municipio,periodo,poblacion,poblacion_mayor_65,poblacion_extranjera,porcentaje_mayor_65,porcentaje_extranjeros
6,Alegría-Dulantzi,2003,1707.0,181.0,30.0,10.603398,1.757469
7,Alegría-Dulantzi,2004,1919.0,174.0,56.0,9.067223,2.918187
8,Alegría-Dulantzi,2005,2048.0,176.0,79.0,8.593750,3.857422
9,Alegría-Dulantzi,2006,2189.0,190.0,92.0,8.679762,4.202832
10,Alegría-Dulantzi,2007,2305.0,203.0,114.0,8.806941,4.945770


## 15. Crear cambio de población por municipio

In [31]:
df = df.sort_values(['codigo_municipio', 'periodo'])

df['variacion_poblacion'] = df.groupby('codigo_municipio')['poblacion'].diff()

In [32]:
#Comprobación
df[['municipio', 'periodo', 'poblacion', 'variacion_poblacion']].head(10)

,municipio,periodo,poblacion,variacion_poblacion
6,Alegría-Dulantzi,2003,1707.0,NaN
7,Alegría-Dulantzi,2004,1919.0,212.0
8,Alegría-Dulantzi,2005,2048.0,129.0
9,Alegría-Dulantzi,2006,2189.0,141.0
10,Alegría-Dulantzi,2007,2305.0,116.0
11,Alegría-Dulantzi,2008,2467.0,162.0
12,Alegría-Dulantzi,2009,2620.0,153.0
13,Alegría-Dulantzi,2010,2714.0,94.0
14,Alegría-Dulantzi,2011,2803.0,89.0
15,Alegría-Dulantzi,2012,2869.0,66.0


## 16. Variación porcentual de población

In [33]:
# Nos permite comparar municipios grandes y pequeños
df['variacion_poblacion_pct'] = df.groupby('codigo_municipio')['poblacion'].pct_change() * 100

In [36]:
# Comprobación
df[[
    'municipio',
    'periodo',
    'poblacion',
    'variacion_poblacion',
    'variacion_poblacion_pct'
]].head(20)

,municipio,periodo,poblacion,variacion_poblacion,variacion_poblacion_pct
6,Alegría-Dulantzi,2003,1707.0,NaN,NaN
7,Alegría-Dulantzi,2004,1919.0,212.0,12.419449
8,Alegría-Dulantzi,2005,2048.0,129.0,6.722251
9,Alegría-Dulantzi,2006,2189.0,141.0,6.884766
10,Alegría-Dulantzi,2007,2305.0,116.0,5.299223
11,Alegría-Dulantzi,2008,2467.0,162.0,7.028200
12,Alegría-Dulantzi,2009,2620.0,153.0,6.201865
13,Alegría-Dulantzi,2010,2714.0,94.0,3.587786
14,Alegría-Dulantzi,2011,2803.0,89.0,3.279293
15,Alegría-Dulantzi,2012,2869.0,66.0,2.354620


## 17. Redondeamos variables derivadas

In [39]:
df['densidad_poblacion'] = df['densidad_poblacion'].round(2)
df['porcentaje_mayor_65'] = df['porcentaje_mayor_65'].round(2)
df['porcentaje_extranjeros'] = df['porcentaje_extranjeros'].round(2)
df['variacion_poblacion'] = df['variacion_poblacion'].round(2)

In [40]:
df.head()

,codigo_municipio,municipio,periodo,poblacion,poblacion_mayor_65,poblacion_extranjera,superficie_km2,densidad_poblacion,porcentaje_mayor_65,porcentaje_extranjeros,variacion_poblacion,variacion_poblacion_pct
6,1001,Alegría-Dulantzi,2003,1707.0,181.0,30.0,19.95,85.56,10.60,1.76,NaN,NaN
7,1001,Alegría-Dulantzi,2004,1919.0,174.0,56.0,19.95,96.19,9.07,2.92,212.0,12.419449
8,1001,Alegría-Dulantzi,2005,2048.0,176.0,79.0,19.95,102.66,8.59,3.86,129.0,6.722251
9,1001,Alegría-Dulantzi,2006,2189.0,190.0,92.0,19.95,109.72,8.68,4.20,141.0,6.884766
10,1001,Alegría-Dulantzi,2007,2305.0,203.0,114.0,19.95,115.54,8.81,4.95,116.0,5.299223


## 18. Guardar dataset final

In [41]:
df.to_csv("../data_processed/dataset_maestro_municipios.csv", index=False)